# Phase 2, Experiment 2: rhythmic context

This notebook compares the phase 2 GTZAN CNN with longer windows and a tempogram input channel. It keeps the repository's fixed Sturm split manifests and baseline training recipe. Track predictions are the mean of the window softmax probabilities.

## Assumptions and baseline reference

- The phase 2 baseline is the source of truth for implementation: PyTorch; 22,050 Hz mono; first 30 seconds; 128-bin log-mel (2,048 FFT, 512-sample hop, power-to-dB with an 80 dB range); 5-second, non-overlapping windows; AdamW at 1e-3 with 1e-4 weight decay; batch 32; 30 epochs; ReduceLROnPlateau and early stopping after 6 validation epochs; seed 42. The best checkpoint is selected by validation track accuracy, with macro-F1 as a tie-breaker.
- The baseline notebook's saved output is 0.645 window accuracy and 0.641 window macro-F1, while the requested reference is 0.650 and 0.636. The reproduction gate compares against the requested numbers and prints a prominent mismatch if they differ. It will not silently claim reproduction.
- The manifests are used as-is; this notebook never re-splits tracks. The baseline says the paper split keeps artists together. The notebook checks manifest membership and exact-copy leakage, but does not infer artist identity from audio.
- A long-window hop is scaled by the same factor as the window length, preserving the baseline's zero-overlap ratio. Thus the number of windows per track is approximately inversely proportional to the window scale; the exact counts are printed.
- The tempogram is computed from the full-track onset-strength envelope at the mel hop rate. For the CNN's stacked-channel input, its lag axis is linearly resampled to 128 rows and min-max normalized per track. This pragmatic alignment keeps the baseline CNN almost unchanged; a two-branch network with late fusion would model the distinct mel and lag axes more cleanly.
- The supplied rock-to-disco list is the reference for the 16-track table. Artist grouping uses only the ranges named in the task; IDs not covered by those ranges remain unassigned.
- The notebook runs locally or in Colab, uses the repository libraries plus librosa, and automatically selects CUDA when PyTorch exposes a GPU; otherwise it runs on CPU.

## Success criteria (set before results)

Primary: the mean rock-to-disco count drops by more than the seed-to-seed spread. Guardrails: mean track macro-F1 does not drop, and no other directed confusion grows by more than its seed-to-seed spread. QUICK_RUN is only a smoke run and cannot establish these criteria.

In [ ]:
# One config cell for paths, windows, seeds, epochs, and smoke runs.
from pathlib import Path
import hashlib
import json
import math
import random
import shutil
import tarfile
from urllib.request import urlopen

QUICK_RUN = False
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "filtered_splits").exists() and (PROJECT_ROOT.parent / "filtered_splits").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "genres_original"
if not DATA_DIR.exists():
    DATA_DIR = next(
        (path for path in [
            PROJECT_ROOT / "genres_original",
            PROJECT_ROOT / "Data" / "genres_original",
            PROJECT_ROOT / "data" / "Data" / "genres_original",
        ] if path.exists()),
        DATA_DIR,
    )
SPLIT_MANIFEST_DIR = PROJECT_ROOT / "filtered_splits"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
RESULTS_DIR = ARTIFACTS_DIR / "phase_2_experiment_2"

BASELINE_SEED = 42
SEEDS = [42, 43, 44]             # Includes the baseline's original seed.
BASELINE_WINDOW_SECONDS = 5.0
LONG_WINDOW_MULTIPLIERS = [2]    # For example: [2, 4].
VIZ_TRACK = "rock.00039"
EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
REUSE_SAVED_RESULTS = True
ALLOW_DATA_DOWNLOAD = True

if QUICK_RUN:
    SEEDS = [BASELINE_SEED]
    EPOCHS = 3

SAMPLE_RATE = 22_050
TRACK_SECONDS = 30
N_MELS = 128
N_FFT = 2_048
HOP_LENGTH = 512
DB_RANGE = 80.0
TEMP_WIN_SECONDS = 8.0
EXPECTED_SAMPLES = SAMPLE_RATE * TRACK_SECONDS
EXPECTED_FRAMES = 1 + EXPECTED_SAMPLES // HOP_LENGTH
BASELINE_WINDOW_FRAMES = math.ceil(BASELINE_WINDOW_SECONDS * SAMPLE_RATE / HOP_LENGTH)
BASELINE_HOP_FRAMES = BASELINE_WINDOW_FRAMES
DEVICE = None  # Set below after importing torch.

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_CACHE_DIR = (
    ARTIFACTS_DIR / "gtzan_log_mel" /
    f"sr{SAMPLE_RATE}_mels{N_MELS}_fft{N_FFT}_hop{HOP_LENGTH}_sec{TRACK_SECONDS}_db{int(DB_RANGE)}"
)
TEMP_CACHE_DIR = (
    ARTIFACTS_DIR / "gtzan_tempogram" /
    f"sr{SAMPLE_RATE}_mels{N_MELS}_fft{N_FFT}_hop{HOP_LENGTH}_sec{TRACK_SECONDS}_win{int(TEMP_WIN_SECONDS)}"
)
FEATURE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

import librosa
import librosa.display
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(BASELINE_SEED)
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
print(f"Data: {DATA_DIR.resolve()}")
print(f"Artifacts: {ARTIFACTS_DIR.resolve()}")
print(f"Seeds: {SEEDS} | epochs: {EPOCHS} | QUICK_RUN: {QUICK_RUN}")

## Data and fixed splits

This follows the baseline's audio indexing and fixed manifest workflow. If GTZAN is not already in the configured data folder, the standard archive is downloaded and extracted with Python's standard library. The filtered split files are read locally or fetched from the same project repository; no new split is created.

In [ ]:
GENRE_FOLDERS = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]

def genre_file_counts(root):
    return {genre: len(list((root / genre).glob("*.wav"))) for genre in GENRE_FOLDERS}

def download_gtzan_if_needed():
    if all(count >= 90 for count in genre_file_counts(DATA_DIR).values()):
        print(f"GTZAN already present at {DATA_DIR.resolve()}")
        return
    if not ALLOW_DATA_DOWNLOAD:
        raise FileNotFoundError(f"GTZAN is incomplete at {DATA_DIR}; enable ALLOW_DATA_DOWNLOAD.")
    archive_path = ARTIFACTS_DIR / "genres.tar.gz"
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if not archive_path.exists():
        url = "https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz"
        print(f"Downloading GTZAN archive from {url}")
        with urlopen(url, timeout=60) as response, archive_path.open("wb") as target:
            shutil.copyfileobj(response, target)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    extracted = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in archive.getmembers():
            if not member.isfile():
                continue
            parts = Path(member.name).parts
            if Path(member.name).is_absolute() or ".." in parts:
                raise ValueError(f"Unsafe path in GTZAN archive: {member.name}")
            filename = Path(member.name).name
            if not filename.lower().endswith(".wav") or filename.startswith("._"):
                continue
            genres = [part.lower() for part in parts[:-1] if part.lower() in GENRE_FOLDERS]
            if not genres:
                continue
            genre = genres[-1]
            if genre == "jazz" and filename == "jazz.00054.wav":
                continue
            destination = DATA_DIR / genre / filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            extracted += 1
    counts = genre_file_counts(DATA_DIR)
    if any(count < 90 for count in counts.values()):
        raise RuntimeError(f"GTZAN extraction is incomplete: {counts}")
    print(f"Extracted {extracted} usable WAV files. Per-genre counts: {counts}")

download_gtzan_if_needed()
if not DATA_DIR.exists():
    alternatives = [
        Path("genres_original"),
        Path("Data/genres_original"),
        Path("data/Data/genres_original"),
    ]
    DATA_DIR = next((path for path in alternatives if path.exists()), DATA_DIR)
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Point DATA_DIR at the folder containing the ten genre folders: {DATA_DIR}")

genre_dirs = sorted(path for path in DATA_DIR.iterdir() if path.is_dir())
CLASS_NAMES = [path.name for path in genre_dirs]
if len(CLASS_NAMES) != 10:
    raise ValueError(f"Expected ten genre folders, found {len(CLASS_NAMES)}: {CLASS_NAMES}")
class_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as audio_file:
        for block in iter(lambda: audio_file.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()

records = []
audio_extensions = {".wav", ".au", ".mp3", ".flac", ".ogg"}
for genre_dir in genre_dirs:
    for audio_path in sorted(genre_dir.iterdir()):
        if not audio_path.is_file() or audio_path.suffix.lower() not in audio_extensions:
            continue
        try:
            info = sf.info(str(audio_path))
            records.append({
                "path": audio_path.resolve(),
                "relative_path": audio_path.relative_to(DATA_DIR).as_posix(),
                "track_name": audio_path.name,
                "genre": genre_dir.name,
                "label_id": class_to_id[genre_dir.name],
                "duration_seconds": info.duration,
                "sha256": sha256_file(audio_path),
            })
        except (OSError, RuntimeError, ValueError) as error:
            print(f"Skipping unreadable file {audio_path}: {error}")
tracks = pd.DataFrame(records)
if tracks.empty:
    raise ValueError(f"No audio files found under {DATA_DIR}")
print(f"Indexed {len(tracks)} audio files across {tracks['genre'].nunique()} genres.")
display(tracks.groupby("genre").size().rename("tracks").to_frame())

### Apply the repository's manifests

The manifest paths, not a random split, define train, validation, and test. This retains the intended 48% / 21% / 31% Sturm split. Exact byte copies are checked so they cannot cross split boundaries, matching the baseline's data-quality checks.

In [ ]:
MANIFEST_FILES = {
    "train": "train_filtered.txt",
    "validation": "valid_filtered.txt",
    "test": "test_filtered.txt",
}
MANIFEST_BASE_URL = "https://raw.githubusercontent.com/Rani1281/SongGenreClassification/main/filtered_splits"

def read_manifest(filename):
    manifest_path = SPLIT_MANIFEST_DIR / filename
    if not manifest_path.exists():
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        with urlopen(f"{MANIFEST_BASE_URL}/{filename}", timeout=30) as response:
            manifest_path.write_bytes(response.read())
    entries = []
    for line_number, line in enumerate(manifest_path.read_text(encoding="utf-8-sig").splitlines(), 1):
        entry = line.strip().replace("\\", "/")
        if not entry or entry.startswith("#") or entry == "jazz/jazz.00054.wav":
            continue
        parts = entry.split("/")
        if len(parts) != 2 or any(part in {"", ".", ".."} for part in parts):
            raise ValueError(f"Invalid path in {manifest_path}, line {line_number}: {line!r}")
        entries.append("/".join(parts))
    if not entries or len(entries) != len(set(entries)):
        raise ValueError(f"Empty or repeated paths in {manifest_path}")
    return entries

manifest_entries = {split: read_manifest(name) for split, name in MANIFEST_FILES.items()}
path_to_split = {}
for split, entries in manifest_entries.items():
    for relative_path in entries:
        if relative_path in path_to_split:
            raise ValueError(f"{relative_path} occurs in multiple split manifests.")
        path_to_split[relative_path] = split
available_paths = set(tracks["relative_path"])
missing = sorted(set(path_to_split) - available_paths)
if missing:
    raise FileNotFoundError(f"{len(missing)} manifest tracks are missing. Examples: {missing[:10]}")
tracks["split"] = tracks["relative_path"].map(path_to_split)
tracks = tracks.loc[tracks["split"].notna()].copy().reset_index(drop=True)

selected_conflicts = tracks.groupby("sha256")["genre"].nunique()
selected_conflicts = selected_conflicts[selected_conflicts > 1].index
if len(selected_conflicts):
    display(tracks.loc[tracks["sha256"].isin(selected_conflicts),
                       ["relative_path", "genre", "split", "sha256"]])
    raise ValueError("The selected manifests contain exact-content copies with conflicting labels.")
hash_split_counts = tracks.groupby("sha256")["split"].nunique()
if (hash_split_counts > 1).any():
    raise ValueError("An exact-content copy crosses the fixed split boundary.")

split_counts = pd.crosstab(tracks["genre"], tracks["split"]).reindex(
    index=CLASS_NAMES, columns=["train", "validation", "test"], fill_value=0
)
if (split_counts == 0).any().any():
    raise ValueError(f"Every genre must occur in every split:\n{split_counts}")
display(split_counts)
print("Split totals:", split_counts.sum(axis=0).to_dict())
print("Percentages:", (100 * split_counts.sum(axis=0) / split_counts.values.sum()).round(1).to_dict())
split_signature = hashlib.sha256(
    json.dumps({k: sorted(v) for k, v in manifest_entries.items()}, sort_keys=True).encode()
).hexdigest()[:12]

train_tracks = tracks.loc[tracks["split"] == "train"].reset_index(drop=True)
validation_tracks = tracks.loc[tracks["split"] == "validation"].reset_index(drop=True)
test_tracks = tracks.loc[tracks["split"] == "test"].reset_index(drop=True)

## Full-track feature extraction and cache

The baseline's log-mel extraction is retained. The experiment additionally computes onset strength and a librosa tempogram on the complete 30-second waveform before slicing any windows. This avoids introducing artificial onset or tempogram boundaries at each crop. Mel arrays use the baseline's on-disk cache path; tempograms use a separate cache keyed by feature settings.

In [ ]:
def feature_path_for_hash(file_hash):
    return FEATURE_CACHE_DIR / f"{file_hash}.npy"

def tempogram_path_for_hash(file_hash):
    return TEMP_CACHE_DIR / f"{file_hash}.npz"

def fit_frames(features, frame_count):
    if features.shape[1] < frame_count:
        return np.pad(features, ((0, 0), (0, frame_count - features.shape[1])))
    return features[:, :frame_count]

def resample_lag_axis(tempogram, target_rows):
    old_axis = np.linspace(0.0, 1.0, tempogram.shape[0])
    new_axis = np.linspace(0.0, 1.0, target_rows)
    return np.stack(
        [np.interp(new_axis, old_axis, tempogram[:, frame])
         for frame in range(tempogram.shape[1])],
        axis=1,
    )

def extract_full_track_features(audio_path):
    audio, _ = librosa.load(
        str(audio_path), sr=SAMPLE_RATE, mono=True, duration=TRACK_SECONDS
    )
    if len(audio) < EXPECTED_SAMPLES:
        audio = np.pad(audio, (0, EXPECTED_SAMPLES - len(audio)))
    else:
        audio = audio[:EXPECTED_SAMPLES]

    # This is the phase 2 baseline's log-mel transform and scaling.
    power_mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, power=2.0,
    )
    decibels = librosa.power_to_db(power_mel, ref=np.max, top_db=DB_RANGE)
    mel = np.clip((decibels + DB_RANGE) / DB_RANGE, 0.0, 1.0).astype(np.float32)
    mel = fit_frames(mel, EXPECTED_FRAMES).astype(np.float32)

    onset_envelope = librosa.onset.onset_strength(
        y=audio, sr=SAMPLE_RATE, hop_length=HOP_LENGTH, n_fft=N_FFT
    )
    onset_envelope = np.pad(
        onset_envelope[:EXPECTED_FRAMES],
        (0, max(0, EXPECTED_FRAMES - len(onset_envelope))),
    )
    tempogram = librosa.feature.tempogram(
        onset_envelope=onset_envelope, sr=SAMPLE_RATE,
        hop_length=HOP_LENGTH,
        win_length=max(2, round(TEMP_WIN_SECONDS * SAMPLE_RATE / HOP_LENGTH)),
    )
    tempogram = fit_frames(tempogram.astype(np.float32), EXPECTED_FRAMES)
    model_tempogram = resample_lag_axis(tempogram, N_MELS)
    low, high = float(model_tempogram.min()), float(model_tempogram.max())
    if high > low:
        model_tempogram = (model_tempogram - low) / (high - low)
    else:
        model_tempogram = np.zeros_like(model_tempogram)
    return mel, tempogram.astype(np.float32), model_tempogram.astype(np.float16)

unique_tracks = tracks.drop_duplicates("sha256")
for position, row in enumerate(unique_tracks.itertuples(index=False), 1):
    mel_path = feature_path_for_hash(row.sha256)
    temp_path = tempogram_path_for_hash(row.sha256)
    missing_mel, missing_temp = not mel_path.exists(), not temp_path.exists()
    if missing_mel or missing_temp:
        mel, raw_tempogram, model_tempogram = extract_full_track_features(row.path)
        if missing_mel:
            np.save(mel_path, mel)
        if missing_temp:
            np.savez_compressed(temp_path, raw=raw_tempogram, model=model_tempogram)
    if position % 100 == 0 or position == len(unique_tracks):
        print(f"Cached full-track features for {position}/{len(unique_tracks)} unique tracks")

print(f"Log-mel cache: {FEATURE_CACHE_DIR.resolve()}")
print(f"Tempogram cache: {TEMP_CACHE_DIR.resolve()}")
sample_hash = unique_tracks.iloc[0]["sha256"]
print("Cached feature shapes:", np.load(feature_path_for_hash(sample_hash), mmap_mode="r").shape,
      np.load(tempogram_path_for_hash(sample_hash))["raw"].shape)

## Feature visualization: one full track

The reference track is rock.00039, one of the supplied rock-to-disco errors. Its expected baseline label is disco; the gate below checks the actual reproduction result. The main figure uses the full track and shows the first baseline and default long-window spans. The small figure compares the raw first-window tempogram with the exact resized and normalized channel supplied to the CNN.

In [ ]:
viz_matches = tracks.loc[tracks["track_name"].str.replace(".wav", "", regex=False) == VIZ_TRACK]
if viz_matches.empty:
    raise KeyError(f"VIZ_TRACK {VIZ_TRACK!r} was not found in the fixed manifests.")
viz_row = viz_matches.iloc[0]
viz_mel = np.load(feature_path_for_hash(viz_row["sha256"]), mmap_mode="r")
viz_feature_file = np.load(tempogram_path_for_hash(viz_row["sha256"]))
viz_raw_temp = viz_feature_file["raw"]
viz_model_temp = viz_feature_file["model"].astype(np.float32)
viz_true_label = viz_row["genre"]
viz_expected_prediction = "disco" if VIZ_TRACK == "rock.00039" else "not yet checked"
default_long_seconds = BASELINE_WINDOW_SECONDS * LONG_WINDOW_MULTIPLIERS[0]
time_seconds = np.arange(EXPECTED_FRAMES) * HOP_LENGTH / SAMPLE_RATE

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
mel_image = librosa.display.specshow(
    viz_mel, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
    x_axis="time", y_axis="mel", ax=axes[0], cmap="magma",
)
axes[0].set_title("Log-mel spectrogram")
axes[0].set_ylabel("Mel frequency (mel)")
axes[0].set_xlabel("Time (s)")
fig.colorbar(mel_image, ax=axes[0], label="Scaled log-mel energy (0–1)")

temp_image = librosa.display.specshow(
    viz_raw_temp, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
    x_axis="time", y_axis="tempo", ax=axes[1], cmap="viridis",
)
axes[1].set_title("Raw full-track tempogram")
axes[1].set_ylabel("Tempo (beats/min)")
axes[1].set_xlabel("Time (s)")
fig.colorbar(temp_image, ax=axes[1], label="Tempogram strength (a.u.)")

for ax in axes:
    ax.set_xlim(0, TRACK_SECONDS)
    ax.axvspan(0, BASELINE_WINDOW_SECONDS, color="cyan", alpha=0.16)
    ax.axvspan(0, default_long_seconds, color="orange", alpha=0.12)
axes[0].legend(handles=[
    Patch(facecolor="cyan", alpha=0.25, label=f"Baseline window: {BASELINE_WINDOW_SECONDS:g} s"),
    Patch(facecolor="orange", alpha=0.25, label=f"LONG_WINDOW: {default_long_seconds:g} s"),
], loc="upper right")
fig.suptitle(
    f"{VIZ_TRACK} | true: {viz_true_label} | baseline prediction (reference): "
    f"{viz_expected_prediction}",
    y=1.02,
)
fig.tight_layout()
main_viz_path = RESULTS_DIR / "feature_visualization_full_track.png"
fig.savefig(main_viz_path, dpi=160, bbox_inches="tight")
plt.show()

viz_window_frames = BASELINE_WINDOW_FRAMES
viz_window_end = min(EXPECTED_FRAMES, viz_window_frames)
raw_window = viz_raw_temp[:, :viz_window_end]
model_window = viz_model_temp[:, :viz_window_end]
window_times = time_seconds[:viz_window_end]
fig_small, small_axes = plt.subplots(1, 2, figsize=(12, 4))
raw_img = librosa.display.specshow(
    raw_window, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
    x_axis="time", y_axis="tempo", ax=small_axes[0], cmap="viridis",
)
small_axes[0].set_title(f"Raw tempogram, first {BASELINE_WINDOW_SECONDS:g} s")
small_axes[0].set_xlabel("Time within window (s)")
small_axes[0].set_ylabel("Tempo (beats/min)")
fig_small.colorbar(raw_img, ax=small_axes[0], label="Tempogram strength (a.u.)")
model_img = small_axes[1].imshow(
    model_window, origin="lower", aspect="auto", interpolation="nearest",
    extent=[window_times[0], window_times[-1], 0, N_MELS], cmap="viridis",
)
small_axes[1].set_title("Tempogram channel received by CNN")
small_axes[1].set_xlabel("Time within window (s)")
small_axes[1].set_ylabel("Resampled lag bin (index)")
fig_small.colorbar(model_img, ax=small_axes[1], label="Normalized value (0–1)")
fig_small.tight_layout()
fig_small.suptitle(f"{VIZ_TRACK} | first baseline window", y=1.04)
small_viz_path = RESULTS_DIR / "feature_visualization_model_input.png"
fig_small.savefig(small_viz_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figures: {main_viz_path} and {small_viz_path}")

Look for a clear tempo ridge in the tempogram and check whether it stays at a similar tempo across the highlighted window. The small comparison shows how the raw lag/BPM information is compressed onto the mel-sized channel grid and normalized before the CNN sees it.

## Cell 1 — baseline reproduction gate

This gate runs before any variant training. It uses seed 42 and the exact baseline window, architecture, optimizer, scheduler, checkpoint selection, and mean-softmax aggregation. A compatible saved experiment checkpoint or the baseline notebook checkpoint may be loaded; otherwise the baseline is retrained. The held-out test predictions are computed once for this run and saved for later analysis.

In [ ]:
class SmallAudioCNN(nn.Module):
    # Same phase 2 CNN; input_channels only changes when the tempogram is enabled.
    def __init__(self, number_of_classes, input_channels=1):
        super().__init__()
        self.filters = nn.Sequential(
            nn.Conv2d(input_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
            nn.Dropout(p=0.30), nn.Linear(128, number_of_classes),
        )

    def forward(self, inputs):
        return self.classifier(self.filters(inputs))

class WindowDataset(Dataset):
    def __init__(self, track_table, window_frames, hop_frames, use_tempogram):
        self.track_rows = track_table.to_dict(orient="records")
        self.window_frames = int(window_frames)
        self.hop_frames = int(hop_frames)
        self.use_tempogram = use_tempogram
        self.examples = [
            (track_index, start)
            for track_index, row in enumerate(self.track_rows)
            for start in range(0, EXPECTED_FRAMES, self.hop_frames)
        ]
        self._temp_cache = {}

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, item_index):
        track_index, start = self.examples[item_index]
        row = self.track_rows[track_index]
        mel = np.load(feature_path_for_hash(row["sha256"]), mmap_mode="r")
        window = np.array(mel[:, start:start + self.window_frames], dtype=np.float32, copy=True)
        if window.shape[1] < self.window_frames:
            window = np.pad(window, ((0, 0), (0, self.window_frames - window.shape[1])))
        if self.use_tempogram:
            if row["sha256"] not in self._temp_cache:
                with np.load(tempogram_path_for_hash(row["sha256"])) as cached:
                    self._temp_cache[row["sha256"]] = cached["model"].astype(np.float32)
            temp = self._temp_cache[row["sha256"]][:, start:start + self.window_frames]
            if temp.shape[1] < self.window_frames:
                temp = np.pad(temp, ((0, 0), (0, self.window_frames - temp.shape[1])))
            window = np.stack((window, temp), axis=0)
        else:
            window = window[None, :, :]
        return (
            torch.from_numpy(window),
            torch.tensor(row["label_id"], dtype=torch.long),
            torch.tensor(track_index, dtype=torch.long),
        )

def make_loaders(seed, window_frames, hop_frames, use_tempogram):
    train_ds = WindowDataset(train_tracks, window_frames, hop_frames, use_tempogram)
    val_ds = WindowDataset(validation_tracks, window_frames, hop_frames, use_tempogram)
    test_ds = WindowDataset(test_tracks, window_frames, hop_frames, use_tempogram)
    generator = torch.Generator().manual_seed(seed)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, generator=generator)
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return (train_ds, train_dl), (val_ds, val_dl), (test_ds, test_dl)

@torch.inference_mode()
def collect_predictions(model, data_loader, dataset):
    model.eval()
    probabilities, window_labels, window_track_indices = [], [], []
    for features, labels, track_indices in data_loader:
        logits = model(features.to(DEVICE))
        probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
        window_labels.append(labels.numpy())
        window_track_indices.append(track_indices.numpy())
    window_probabilities = np.concatenate(probabilities)
    window_true = np.concatenate(window_labels)
    window_track_indices = np.concatenate(window_track_indices)
    track_probabilities = np.zeros((len(dataset.track_rows), len(CLASS_NAMES)), dtype=np.float64)
    window_counts = np.zeros(len(dataset.track_rows), dtype=np.int64)
    np.add.at(track_probabilities, window_track_indices, window_probabilities)
    np.add.at(window_counts, window_track_indices, 1)
    track_probabilities /= window_counts[:, None]
    track_true = np.asarray([row["label_id"] for row in dataset.track_rows], dtype=np.int64)
    return {
        "window_probabilities": window_probabilities,
        "window_true": window_true,
        "window_predicted": window_probabilities.argmax(axis=1),
        "window_track_indices": window_track_indices,
        "track_probabilities": track_probabilities,
        "track_true": track_true,
        "track_predicted": track_probabilities.argmax(axis=1),
        "track_ids": np.asarray([row["track_name"].rsplit(".", 1)[0] for row in dataset.track_rows]),
    }

def metric_summary(result):
    return {
        "window_accuracy": float(accuracy_score(result["window_true"], result["window_predicted"])),
        "window_macro_f1": float(f1_score(result["window_true"], result["window_predicted"],
                                          average="macro", zero_division=0)),
        "track_accuracy": float(accuracy_score(result["track_true"], result["track_predicted"])),
        "track_macro_f1": float(f1_score(result["track_true"], result["track_predicted"],
                                         average="macro", zero_division=0)),
    }

def result_cache_paths(config_name, seed, signature):
    stem = f"{config_name}__seed{seed}__{signature}"
    return RESULTS_DIR / f"{stem}.npz", RESULTS_DIR / f"{stem}.pt"

def load_prediction_cache(path):
    with np.load(path, allow_pickle=False) as saved:
        return {key: saved[key] for key in saved.files}

def save_prediction_cache(path, result):
    np.savez_compressed(path, **result)

def fit_baseline_or_variant(config_name, seed, window_frames, hop_frames, use_tempogram,
                            signature, permit_legacy_baseline=False):
    prediction_path, checkpoint_path = result_cache_paths(config_name, seed, signature)
    if REUSE_SAVED_RESULTS and prediction_path.exists():
        baseline_model_reloaded = False
        if config_name == "BASELINE" and seed == BASELINE_SEED:
            cached_model = None
            if checkpoint_path.exists():
                candidate = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
                if candidate.get("signature") == signature:
                    cached_model = candidate
            if cached_model is None and permit_legacy_baseline:
                legacy_path = ARTIFACTS_DIR / "gtzan_cnn_baseline.pt"
                if legacy_path.exists():
                    candidate = torch.load(legacy_path, map_location=DEVICE, weights_only=False)
                    legacy_matches = (
                        candidate.get("class_names") == CLASS_NAMES
                        and candidate.get("sample_rate") == SAMPLE_RATE
                        and candidate.get("track_seconds") == TRACK_SECONDS
                        and candidate.get("segment_seconds") == BASELINE_WINDOW_SECONDS
                        and candidate.get("n_mels") == N_MELS
                        and candidate.get("n_fft") == N_FFT
                        and candidate.get("hop_length") == HOP_LENGTH
                    )
                    if legacy_matches:
                        cached_model = candidate
            if cached_model is not None:
                reloaded = SmallAudioCNN(len(CLASS_NAMES), input_channels=1).to(DEVICE)
                reloaded.load_state_dict(cached_model["model_state_dict"])
                baseline_model_reloaded = True
                print("Reloaded the baseline checkpoint; reusing its cached one-time test predictions.")
            else:
                print("Baseline prediction cache has no compatible checkpoint; retraining the baseline.")
        else:
            result = load_prediction_cache(prediction_path)
            return result, "cached predictions"
        if baseline_model_reloaded:
            result = load_prediction_cache(prediction_path)
            return result, "reloaded checkpoint and cached predictions"

    (train_ds, train_dl), (val_ds, val_dl), (test_ds, test_dl) = make_loaders(
        seed, window_frames, hop_frames, use_tempogram
    )
    set_seed(seed)
    if config_name == "BASELINE" and seed == BASELINE_SEED:
        # Match the reference notebook's sample batch and pre-training model cell.
        _baseline_sample_batch = next(iter(train_dl))
        _reference_notebook_model = SmallAudioCNN(
            len(CLASS_NAMES), input_channels=1
        ).to(DEVICE)
    model = SmallAudioCNN(len(CLASS_NAMES), input_channels=2 if use_tempogram else 1).to(DEVICE)
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loaded_checkpoint = False
    best_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        if checkpoint.get("signature") == signature:
            model.load_state_dict(checkpoint["model_state_dict"])
            best_epoch = int(checkpoint.get("best_epoch", 0))
            loaded_checkpoint = True
            print(f"Reloaded {config_name}, seed {seed}, best epoch {best_epoch}.")
    legacy_path = ARTIFACTS_DIR / "gtzan_cnn_baseline.pt"
    if (not loaded_checkpoint and permit_legacy_baseline and legacy_path.exists()):
        legacy = torch.load(legacy_path, map_location=DEVICE, weights_only=False)
        matching = (
            legacy.get("class_names") == CLASS_NAMES
            and legacy.get("sample_rate") == SAMPLE_RATE
            and legacy.get("track_seconds") == TRACK_SECONDS
            and legacy.get("segment_seconds") == BASELINE_WINDOW_SECONDS
            and legacy.get("n_mels") == N_MELS
            and legacy.get("n_fft") == N_FFT
            and legacy.get("hop_length") == HOP_LENGTH
        )
        if matching:
            model.load_state_dict(legacy["model_state_dict"])
            loaded_checkpoint = True
            print(f"Loaded compatible reference checkpoint {legacy_path}.")

    if not loaded_checkpoint:
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=2
        )
        best_score, best_state, epochs_without_improvement = (-1.0, -1.0), None, 0
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss, correct_windows, seen_windows = 0.0, 0, 0
            for features, labels, _ in train_dl:
                features, labels = features.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                logits = model(features)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
                batch_size = labels.size(0)
                total_loss += loss.item() * batch_size
                correct_windows += (logits.argmax(dim=1) == labels).sum().item()
                seen_windows += batch_size
            validation = collect_predictions(model, val_dl, val_ds)
            val_accuracy = accuracy_score(validation["track_true"], validation["track_predicted"])
            val_macro_f1 = f1_score(validation["track_true"], validation["track_predicted"],
                                    average="macro", zero_division=0)
            scheduler.step(val_accuracy)
            score = (val_accuracy, val_macro_f1)
            if score > best_score:
                best_score, best_epoch, epochs_without_improvement = score, epoch, 0
                best_state = {name: value.detach().cpu().clone()
                              for name, value in model.state_dict().items()}
                torch.save({
                    "model_state_dict": best_state,
                    "signature": signature,
                    "class_names": CLASS_NAMES,
                    "sample_rate": SAMPLE_RATE,
                    "track_seconds": TRACK_SECONDS,
                    "window_frames": window_frames,
                    "hop_frames": hop_frames,
                    "n_mels": N_MELS,
                    "n_fft": N_FFT,
                    "hop_length": HOP_LENGTH,
                    "use_tempogram": use_tempogram,
                    "best_epoch": best_epoch,
                }, checkpoint_path)
            else:
                epochs_without_improvement += 1
            print(
                f"{config_name} seed {seed} | epoch {epoch:02d}/{EPOCHS} | "
                f"loss {total_loss / seen_windows:.4f} | train window acc "
                f"{correct_windows / seen_windows:.3f} | val track acc {val_accuracy:.3f} | "
                f"val macro-F1 {val_macro_f1:.3f}"
            )
            if epochs_without_improvement >= PATIENCE:
                print(f"Early stopping after {PATIENCE} epochs without validation improvement.")
                break
        if best_state is None:
            raise RuntimeError(f"{config_name} seed {seed} produced no validation checkpoint.")
        model.load_state_dict(best_state)

    # The test loader is touched exactly once for this trained/reloaded model.
    result = collect_predictions(model, test_dl, test_ds)
    save_prediction_cache(prediction_path, result)
    print(f"Best validation epoch: {best_epoch or 'from compatible legacy checkpoint'}")
    print(f"Trainable parameters: {params:,}")
    return result, ("reloaded checkpoint" if loaded_checkpoint else "trained")

BASELINE_SIGNATURE_DATA = {
    "config": "BASELINE", "seed": BASELINE_SEED, "window_frames": BASELINE_WINDOW_FRAMES,
    "hop_frames": BASELINE_HOP_FRAMES, "use_tempogram": False,
    "epochs": EPOCHS, "patience": PATIENCE, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "split": split_signature,
    "features": [SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH, DB_RANGE],
}
BASELINE_SIGNATURE = hashlib.sha256(
    json.dumps(BASELINE_SIGNATURE_DATA, sort_keys=True).encode()
).hexdigest()[:12]
baseline_result, baseline_source = fit_baseline_or_variant(
    "BASELINE", BASELINE_SEED, BASELINE_WINDOW_FRAMES, BASELINE_HOP_FRAMES,
    False, BASELINE_SIGNATURE, permit_legacy_baseline=True,
)
baseline_metrics = metric_summary(baseline_result)
baseline_confusion = confusion_matrix(
    baseline_result["track_true"], baseline_result["track_predicted"],
    labels=np.arange(len(CLASS_NAMES)),
)

if QUICK_RUN:
    print("\\nSMOKE RUN ONLY: QUICK_RUN uses 3 epochs and one seed; target reproduction is not expected.")
    baseline_match = False
else:
    expected_checks = [
        ("window accuracy", baseline_metrics["window_accuracy"], 0.650),
        ("window macro-F1", baseline_metrics["window_macro_f1"], 0.636),
    ]
    id_by_name = {name: index for index, name in enumerate(CLASS_NAMES)}
    expected_checks.extend([
        ("rock→disco tracks", int(baseline_confusion[id_by_name["rock"], id_by_name["disco"]]), 16),
        ("pop→disco tracks", int(baseline_confusion[id_by_name["pop"], id_by_name["disco"]]), 13),
        ("blues→jazz tracks", int(baseline_confusion[id_by_name["blues"], id_by_name["jazz"]]), 12),
    ])
    observed_largest = sorted(
        [
            (CLASS_NAMES[true_id], CLASS_NAMES[predicted_id],
             int(baseline_confusion[true_id, predicted_id]))
            for true_id in range(len(CLASS_NAMES))
            for predicted_id in range(len(CLASS_NAMES))
            if true_id != predicted_id
        ],
        key=lambda item: item[2],
        reverse=True,
    )[:3]
    expected_largest = [("rock", "disco", 16), ("pop", "disco", 13), ("blues", "jazz", 12)]
    expected_checks.append(("three largest track confusions", observed_largest, expected_largest))
    check_rows = []
    for name, observed, expected in expected_checks:
        equal = round(observed, 3) == expected if isinstance(expected, float) else observed == expected
        difference = observed - expected if isinstance(observed, (int, float, np.number)) else ("same" if equal else "different")
        check_rows.append({"check": name, "expected": expected, "observed": observed,
                           "difference": difference, "status": "MATCH" if equal else "MISMATCH"})
    check_table = pd.DataFrame(check_rows)
    display(check_table)
    baseline_match = bool((check_table["status"] == "MATCH").all())
    if baseline_match:
        print(f"BASELINE REPRODUCTION MATCHED. Source: {baseline_source}")
    else:
        print("!!! BASELINE REPRODUCTION MISMATCH: inspect the differences above before interpreting variants. !!!")

baseline_prediction_for_viz = CLASS_NAMES[int(baseline_result["track_predicted"][
    np.flatnonzero(baseline_result["track_ids"] == VIZ_TRACK)[0]
])] if np.any(baseline_result["track_ids"] == VIZ_TRACK) else "not present in test result"
fig.suptitle(
    f"{VIZ_TRACK} | true: {viz_true_label} | baseline predicted: {baseline_prediction_for_viz}",
    y=1.02,
)
fig.savefig(main_viz_path, dpi=160, bbox_inches="tight")
print(f"Actual baseline prediction for {VIZ_TRACK}: {baseline_prediction_for_viz}")
print("Baseline test metrics:", baseline_metrics)
print("Baseline track confusion matrix (true rows, predicted columns):")
display(pd.DataFrame(baseline_confusion, index=CLASS_NAMES, columns=CLASS_NAMES))

## Models and configurations

The baseline model has adaptive global average pooling, so changing window duration does not require changing the dense-layer size. The only architecture change is the first convolution's input-channel count (one for log-mel, two for log-mel plus normalized tempogram). The same pooling stack and classifier are retained. Parameter counts are printed for every configuration.

In [ ]:
CONFIGS = [{
    "name": "BASELINE",
    "label": "BASELINE",
    "window_frames": BASELINE_WINDOW_FRAMES,
    "hop_frames": BASELINE_HOP_FRAMES,
    "window_seconds": BASELINE_WINDOW_SECONDS,
    "hop_seconds": BASELINE_HOP_FRAMES * HOP_LENGTH / SAMPLE_RATE,
    "use_tempogram": False,
}]
for multiplier in LONG_WINDOW_MULTIPLIERS:
    window_frames = int(round(BASELINE_WINDOW_FRAMES * multiplier))
    hop_frames = int(round(BASELINE_HOP_FRAMES * multiplier))
    duration = BASELINE_WINDOW_SECONDS * multiplier
    CONFIGS.append({
        "name": f"LONG_WINDOW_{multiplier}x", "label": f"LONG_WINDOW {multiplier}x",
        "window_frames": window_frames, "hop_frames": hop_frames,
        "window_seconds": duration, "hop_seconds": hop_frames * HOP_LENGTH / SAMPLE_RATE,
        "use_tempogram": False,
    })
CONFIGS.append({
    "name": "TEMPOGRAM", "label": "TEMPOGRAM",
    "window_frames": BASELINE_WINDOW_FRAMES, "hop_frames": BASELINE_HOP_FRAMES,
    "window_seconds": BASELINE_WINDOW_SECONDS,
    "hop_seconds": BASELINE_HOP_FRAMES * HOP_LENGTH / SAMPLE_RATE,
    "use_tempogram": True,
})
for multiplier in LONG_WINDOW_MULTIPLIERS:
    window_frames = int(round(BASELINE_WINDOW_FRAMES * multiplier))
    hop_frames = int(round(BASELINE_HOP_FRAMES * multiplier))
    duration = BASELINE_WINDOW_SECONDS * multiplier
    CONFIGS.append({
        "name": f"BOTH_{multiplier}x", "label": f"BOTH {multiplier}x",
        "window_frames": window_frames, "hop_frames": hop_frames,
        "window_seconds": duration, "hop_seconds": hop_frames * HOP_LENGTH / SAMPLE_RATE,
        "use_tempogram": True,
    })

def number_of_windows(hop_frames):
    return len(range(0, EXPECTED_FRAMES, hop_frames))

config_rows = []
for config in CONFIGS:
    model = SmallAudioCNN(len(CLASS_NAMES), input_channels=2 if config["use_tempogram"] else 1)
    config_rows.append({
        "configuration": config["label"],
        "window_seconds": config["window_seconds"],
        "hop_seconds": round(config["hop_seconds"], 3),
        "windows_per_track": number_of_windows(config["hop_frames"]),
        "input_channels": 2 if config["use_tempogram"] else 1,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
    })
display(pd.DataFrame(config_rows))

## Train or reload all configurations

Every run uses the same fixed split, AdamW settings, epoch budget, validation checkpoint rule, and no added augmentation. The baseline seed-42 run above is reused as one of its three seeds. QUICK_RUN uses one seed and three epochs. Test prediction caches let the reporting cells run later without retraining.

In [ ]:
all_results = {"BASELINE": {BASELINE_SEED: baseline_result}}
run_sources = {("BASELINE", BASELINE_SEED): baseline_source}
for config in CONFIGS:
    config_name = config["name"]
    all_results.setdefault(config_name, {})
    for seed in SEEDS:
        if config_name == "BASELINE" and seed == BASELINE_SEED:
            continue  # The reproduction gate already trained/reloaded and evaluated this run.
        signature_data = {
            "config": config_name, "seed": seed,
            "window_frames": config["window_frames"], "hop_frames": config["hop_frames"],
            "use_tempogram": config["use_tempogram"], "epochs": EPOCHS,
            "patience": PATIENCE, "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE, "split": split_signature,
            "features": [SAMPLE_RATE, N_MELS, N_FFT, HOP_LENGTH, DB_RANGE],
        }
        signature = hashlib.sha256(
            json.dumps(signature_data, sort_keys=True).encode()
        ).hexdigest()[:12]
        result, source = fit_baseline_or_variant(
            config_name, seed, config["window_frames"], config["hop_frames"],
            config["use_tempogram"], signature,
        )
        all_results[config_name][seed] = result
        run_sources[(config_name, seed)] = source

print("\\nRun sources:")
for key, source in run_sources.items():
    print(f"{key}: {source}")

## Evaluation

Window metrics are diagnostics; track metrics use mean-softmax aggregation over all windows in a track. The displayed confusion matrices are the mean count per fixed test track across seeds. The variant difference matrix is mean variant confusion minus mean baseline confusion; positive off-diagonal values indicate more errors of that type.

In [ ]:
def confusion_for(result):
    return confusion_matrix(result["track_true"], result["track_predicted"],
                            labels=np.arange(len(CLASS_NAMES)))

def per_class_f1(result):
    return f1_score(result["track_true"], result["track_predicted"],
                    labels=np.arange(len(CLASS_NAMES)), average=None, zero_division=0)

def targeted_metrics(result):
    matrix = confusion_for(result)
    index = {name: i for i, name in enumerate(CLASS_NAMES)}
    rock, disco = index["rock"], index["disco"]
    rock_recall = matrix[rock, rock] / matrix[rock].sum()
    disco_precision = matrix[disco, disco] / matrix[:, disco].sum()
    return {
        "rock_to_disco": int(matrix[rock, disco]),
        "rock_recall": float(rock_recall),
        "disco_precision": float(disco_precision),
        "pop_to_disco": int(matrix[index["pop"], disco]),
        "blues_to_jazz": int(matrix[index["blues"], index["jazz"]]),
    }

metric_rows = []
mean_confusions = {}
mean_f1_by_config = {}
target_by_config = {}
for config in CONFIGS:
    name = config["name"]
    runs = list(all_results[name].values())
    metrics = [metric_summary(result) for result in runs]
    confusions = np.stack([confusion_for(result) for result in runs])
    mean_confusions[name] = confusions.mean(axis=0)
    class_f1 = np.stack([per_class_f1(result) for result in runs])
    mean_f1_by_config[name] = class_f1.mean(axis=0)
    target_metrics = [targeted_metrics(result) for result in runs]
    target_by_config[name] = target_metrics
    row = {"configuration": config["label"], "seeds": len(runs)}
    for key in metrics[0]:
        values = np.asarray([item[key] for item in metrics], dtype=float)
        row[key] = f"{values.mean():.3f} ± {values.std(ddof=1) if len(values)>1 else 0.0:.3f}"
    for key in target_metrics[0]:
        values = np.asarray([item[key] for item in target_metrics], dtype=float)
        row[key] = f"{values.mean():.2f} ± {values.std(ddof=1) if len(values)>1 else 0.0:.2f}"
    metric_rows.append(row)
metrics_table = pd.DataFrame(metric_rows).set_index("configuration")
display(metrics_table)

for config in CONFIGS:
    name = config["name"]
    confusion_table = pd.DataFrame(mean_confusions[name], index=CLASS_NAMES, columns=CLASS_NAMES)
    print(f"{config['label']} — mean track-level confusion matrix across seeds")
    display(confusion_table.round(2))
    confusion_table.to_csv(RESULTS_DIR / f"{name.lower()}_mean_confusion.csv")

baseline_mean_confusion = mean_confusions["BASELINE"]
for config in CONFIGS:
    if config["name"] == "BASELINE":
        continue
    difference = mean_confusions[config["name"]] - baseline_mean_confusion
    bound = max(1.0, float(np.abs(difference).max()))
    fig, ax = plt.subplots(figsize=(8, 7))
    image = ax.imshow(difference, cmap="coolwarm", vmin=-bound, vmax=bound)
    ax.set_xticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{config['label']}: confusion difference vs baseline")
    for row in range(len(CLASS_NAMES)):
        for column in range(len(CLASS_NAMES)):
            ax.text(column, row, f"{difference[row, column]:+.1f}",
                    ha="center", va="center", fontsize=7)
    fig.colorbar(image, ax=ax, label="Mean count difference (tracks)")
    fig.tight_layout()
    path = RESULTS_DIR / f"{config['name'].lower()}_minus_baseline_heatmap.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved {path}")

## Analysis: class scores, targeted tracks, and paired uncertainty

The class comparison shows per-class track F1 across seeds. The rock-to-disco table reports the mean track probability assigned to the true class and the predicted class after averaging probabilities across seeds within each configuration. The paired bootstrap resamples test-track identities and compares each variant with the same-seed baseline predictions.

In [ ]:
# Per-class grouped bar chart (mean F1 across seeds).
labels = [config["label"] for config in CONFIGS]
x = np.arange(len(CLASS_NAMES))
width = 0.8 / len(CONFIGS)
fig, ax = plt.subplots(figsize=(15, 5))
for config_index, config in enumerate(CONFIGS):
    values = mean_f1_by_config[config["name"]]
    ax.bar(x - 0.4 + width / 2 + config_index * width, values,
           width=width, label=config["label"])
ax.set_xticks(x, CLASS_NAMES, rotation=35, ha="right")
ax.set_ylabel("Track-level F1")
ax.set_xlabel("True genre")
ax.set_ylim(0, 1)
ax.set_title("Per-class track F1 by configuration (seed mean)")
ax.legend(ncol=2)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
f1_plot_path = RESULTS_DIR / "per_class_f1_comparison.png"
fig.savefig(f1_plot_path, dpi=160, bbox_inches="tight")
plt.show()

# All requested targeted counts and per-seed standard deviations.
target_rows = []
for config in CONFIGS:
    name = config["name"]
    row = {"configuration": config["label"]}
    for metric in target_by_config[name][0]:
        values = np.asarray([run[metric] for run in target_by_config[name]], dtype=float)
        row[metric] = f"{values.mean():.2f} ± {values.std(ddof=1) if len(values)>1 else 0.0:.2f}"
    target_rows.append(row)
target_table = pd.DataFrame(target_rows).set_index("configuration")
display(target_table)

ROCK_DISCO_IDS = [
    "rock.00028", "rock.00029", "rock.00030", "rock.00031",
    "rock.00033", "rock.00034", "rock.00036", "rock.00037",
    "rock.00039", "rock.00040", "rock.00041", "rock.00043",
    "rock.00044", "rock.00046", "rock.00047", "rock.00086",
]
ROLLING_STONES_IDS = {28, 29, 30, 31, 33, 35, 37}
LED_ZEPPELIN_IDS = set(range(39, 49))
def artist_group(track_id):
    number = int(track_id.rsplit(".", 1)[1])
    if number in ROLLING_STONES_IDS:
        return "Rolling Stones (task range)"
    if number in LED_ZEPPELIN_IDS:
        return "Led Zeppelin (task range)"
    return "Unassigned by task ranges"

track_table = pd.DataFrame({"track_id": ROCK_DISCO_IDS})
track_table["artist_group"] = track_table["track_id"].map(artist_group)
id_to_index = {track_id: index for index, track_id in enumerate(
    baseline_result["track_ids"].tolist()
)}
missing_target_ids = [track_id for track_id in ROCK_DISCO_IDS if track_id not in id_to_index]
if missing_target_ids:
    print("Reference rock→disco tracks missing from this test manifest:", missing_target_ids)
for config in CONFIGS:
    name = config["name"]
    runs = list(all_results[name].values())
    track_probabilities = np.stack([run["track_probabilities"] for run in runs]).mean(axis=0)
    track_predictions = track_probabilities.argmax(axis=1)
    true_id = class_to_id["rock"]
    probs, predictions = [], []
    correct_flips = 0
    for track_id in ROCK_DISCO_IDS:
        if track_id not in id_to_index:
            probs.append(np.nan)
            predictions.append("not in test manifest")
            continue
        position = id_to_index[track_id]
        probability = float(track_probabilities[position, true_id])
        prediction = CLASS_NAMES[int(track_predictions[position])]
        probs.append(probability)
        predictions.append(prediction)
        correct_flips += int(prediction == "rock")
    track_table[f"{config['label']} p(true rock)"] = probs
    track_table[f"{config['label']} prediction"] = predictions
    track_table.attrs.setdefault("flips", {})[config["label"]] = correct_flips
display(track_table.round(4))
print("Corrected tracks among the reference 16 (seed-mean probabilities):")
for label, count in track_table.attrs["flips"].items():
    print(f"{label}: {count}/16")
print("Artist grouping uses only the task-provided ranges; unassigned IDs are not guessed.")
display(track_table.groupby("artist_group").apply(
    lambda group: pd.Series({
        label: int((group[f"{label} prediction"] == "rock").sum())
        for label in labels
    })
))
for label in labels:
    corrected = track_table.loc[track_table[f"{label} prediction"] == "rock"]
    named_groups = corrected.loc[
        corrected["artist_group"] != "Unassigned by task ranges", "artist_group"
    ]
    group_counts = named_groups.value_counts()
    named_total = int(group_counts.sum())
    if named_total:
        dominant_group = str(group_counts.index[0])
        concentration = float(group_counts.iloc[0] / named_total)
        cluster_note = "yes" if concentration >= 0.75 else "no clear single-artist cluster"
        print(
            f"{label}: {named_total} corrected tracks have a supplied artist mapping; "
            f"largest group is {dominant_group} ({group_counts.iloc[0]}/{named_total}); "
            f"clustered by artist: {cluster_note}."
        )
    else:
        print(f"{label}: no corrected tracks have an artist mapping in the task ranges.")

def paired_bootstrap_accuracy_ci(baseline_runs, variant_runs, bootstrap_samples=5000, seed=2026):
    seeds = sorted(set(baseline_runs) & set(variant_runs))
    per_seed_deltas = []
    for run_seed in seeds:
        baseline = baseline_runs[run_seed]
        variant = variant_runs[run_seed]
        per_seed_deltas.append(
            (variant["track_predicted"] == variant["track_true"]).astype(float)
            - (baseline["track_predicted"] == baseline["track_true"]).astype(float)
        )
    paired = np.stack(per_seed_deltas)  # seed × same fixed test tracks
    rng = np.random.default_rng(seed)
    n_tracks = paired.shape[1]
    estimates = np.empty(bootstrap_samples)
    for sample_index in range(bootstrap_samples):
        sampled_tracks = rng.integers(0, n_tracks, size=n_tracks)
        estimates[sample_index] = paired[:, sampled_tracks].mean()
    return float(paired.mean()), tuple(np.quantile(estimates, [0.025, 0.975]))

significance_rows = []
baseline_runs = all_results["BASELINE"]
baseline_accuracy_by_seed = {
    seed: accuracy_score(result["track_true"], result["track_predicted"])
    for seed, result in baseline_runs.items()
}
baseline_rock_disco = {
    seed: targeted_metrics(result)["rock_to_disco"] for seed, result in baseline_runs.items()
}
for config in CONFIGS:
    name = config["name"]
    if name == "BASELINE":
        continue
    delta, ci = paired_bootstrap_accuracy_ci(baseline_runs, all_results[name])
    variant_counts = {seed: targeted_metrics(result)["rock_to_disco"]
                      for seed, result in all_results[name].items()}
    common_seeds = sorted(set(baseline_rock_disco) & set(variant_counts))
    paired_count_drop = np.asarray([
        baseline_rock_disco[seed] - variant_counts[seed] for seed in common_seeds
    ], dtype=float)
    spread = max(
        np.std([baseline_rock_disco[seed] for seed in common_seeds], ddof=1)
        if len(common_seeds) > 1 else 0.0,
        np.std([variant_counts[seed] for seed in common_seeds], ddof=1)
        if len(common_seeds) > 1 else 0.0,
    )
    accuracy_values = np.asarray([
        accuracy_score(all_results[name][seed]["track_true"],
                       all_results[name][seed]["track_predicted"])
        for seed in common_seeds
    ])
    accuracy_spread = max(
        np.std([baseline_accuracy_by_seed[seed] for seed in common_seeds], ddof=1)
        if len(common_seeds) > 1 else 0.0,
        np.std(accuracy_values, ddof=1) if len(common_seeds) > 1 else 0.0,
    )
    significance_rows.append({
        "configuration": config["label"],
        "paired accuracy difference": delta,
        "95% paired track-bootstrap CI": f"[{ci[0]:.3f}, {ci[1]:.3f}]",
        "mean rock→disco drop": float(paired_count_drop.mean()),
        "rock→disco seed spread": spread,
        "drop / seed spread": float(paired_count_drop.mean() / spread) if spread else np.inf,
        "accuracy seed spread": accuracy_spread,
    })
significance_table = pd.DataFrame(significance_rows).set_index("configuration")
display(significance_table.round(3))

## Save analysis outputs

Each run's predictions and selected weights are cached under the results folder. This cell also saves the combined metric table, targeted metrics, confusion matrices, and the 16-track probability table so the analysis can be reviewed without running training again.

In [ ]:
metrics_table.to_csv(RESULTS_DIR / "metrics_by_configuration.csv")
target_table.to_csv(RESULTS_DIR / "targeted_metrics_by_configuration.csv")
track_table.to_csv(RESULTS_DIR / "rock_to_disco_reference_tracks.csv", index=False)
significance_table.to_csv(RESULTS_DIR / "paired_bootstrap_significance.csv")

summary_json = {
    "baseline_reproduction_match": bool(baseline_match),
    "quick_run": bool(QUICK_RUN),
    "split_signature": split_signature,
    "configurations": {},
}
for config in CONFIGS:
    name = config["name"]
    runs = list(all_results[name].values())
    summary_json["configurations"][name] = {
        "seeds": sorted(all_results[name]),
        "metrics": [metric_summary(result) for result in runs],
        "mean_confusion": mean_confusions[name].tolist(),
        "targeted_metrics": [targeted_metrics(result) for result in runs],
    }
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary_json, indent=2), encoding="utf-8")
print(f"Saved metrics, predictions, confusion matrices, and plots under {RESULTS_DIR.resolve()}")

## Conclusion

The conclusion below applies the success criteria stated before training. In QUICK_RUN mode, it will explicitly label the outcome as provisional.

In [ ]:
baseline_mean_f1 = np.mean([
    metric_summary(result)["track_macro_f1"] for result in all_results["BASELINE"].values()
])
baseline_counts = np.asarray([
    targeted_metrics(result)["rock_to_disco"] for result in all_results["BASELINE"].values()
], dtype=float)
rock_i, disco_i = class_to_id["rock"], class_to_id["disco"]
variant_decisions = []

for config in CONFIGS:
    name = config["name"]
    if name == "BASELINE":
        continue
    runs = list(all_results[name].values())
    variant_counts = np.asarray([targeted_metrics(result)["rock_to_disco"] for result in runs],
                                dtype=float)
    variant_mean_count = float(variant_counts.mean())
    variant_spread = float(variant_counts.std(ddof=1)) if len(variant_counts) > 1 else 0.0
    baseline_spread = float(baseline_counts.std(ddof=1)) if len(baseline_counts) > 1 else 0.0
    seed_spread = max(variant_spread, baseline_spread)
    rock_drop = float(baseline_counts.mean() - variant_mean_count)
    f1_mean = float(np.mean([metric_summary(result)["track_macro_f1"] for result in runs]))
    guardrail_f1 = f1_mean >= baseline_mean_f1

    variant_confusions = np.stack([confusion_for(result) for result in runs])
    variant_mean_confusion = variant_confusions.mean(axis=0)
    grows_too_much = []
    for true_class in range(len(CLASS_NAMES)):
        for predicted_class in range(len(CLASS_NAMES)):
            if true_class == predicted_class or (true_class == rock_i and predicted_class == disco_i):
                continue
            baseline_values = np.asarray([
                confusion_for(result)[true_class, predicted_class]
                for result in all_results["BASELINE"].values()
            ], dtype=float)
            variant_values = variant_confusions[:, true_class, predicted_class].astype(float)
            pair_spread = max(
                baseline_values.std(ddof=1) if len(baseline_values) > 1 else 0.0,
                variant_values.std(ddof=1) if len(variant_values) > 1 else 0.0,
            )
            growth = variant_mean_confusion[true_class, predicted_class] - \
                     mean_confusions["BASELINE"][true_class, predicted_class]
            if growth > pair_spread:
                grows_too_much.append((CLASS_NAMES[true_class], CLASS_NAMES[predicted_class],
                                       float(growth), float(pair_spread)))
    primary_pass = rock_drop > seed_spread
    guardrails_pass = guardrail_f1 and not grows_too_much
    variant_decisions.append({
        "name": config["label"], "primary": primary_pass, "guardrails": guardrails_pass,
        "drop": rock_drop, "spread": seed_spread, "f1": f1_mean,
        "other_growths": grows_too_much,
    })

print(f"Baseline mean track macro-F1: {baseline_mean_f1:.3f}")
if QUICK_RUN:
    print("PROVISIONAL ONLY: QUICK_RUN is a smoke run; do not interpret it as hypothesis evidence.")
elif any(item["primary"] and item["guardrails"] for item in variant_decisions):
    winners = [item["name"] for item in variant_decisions if item["primary"] and item["guardrails"]]
    print("SUPPORTED: " + ", ".join(winners) +
          " met the primary rock→disco criterion and both guardrails.")
elif any(item["primary"] for item in variant_decisions):
    partial = [item["name"] for item in variant_decisions if item["primary"]]
    print("PARTLY SUPPORTED: " + ", ".join(partial) +
          " reduced rock→disco beyond the seed spread, but at least one guardrail failed.")
else:
    print("NOT SUPPORTED: no tested variant reduced rock→disco by more than the seed-to-seed spread.")
    print("If the gains are small, artist overlap within the fixed split may be the cause of the "
          "rock→disco block rather than a lack of rhythmic context.")
for item in variant_decisions:
    print(
        f"{item['name']}: rock→disco drop {item['drop']:.2f}, seed spread "
        f"{item['spread']:.2f}, track macro-F1 {item['f1']:.3f}; "
        f"macro-F1 guardrail={'pass' if item['f1'] >= baseline_mean_f1 else 'fail'}; "
        f"other confusion growths beyond spread={len(item['other_growths'])}."
    )
    if item["other_growths"]:
        print("  Largest guardrail exceptions:", item["other_growths"][:5])
if QUICK_RUN:
    print("Full reference baseline metrics were not checked because QUICK_RUN is enabled.")
elif not baseline_match:
    print("Reference reproduction did not match; treat all variant comparisons as provisional.")
print("Conclusion confidence is limited by the fixed split and the possible artist overlap "
      "noted in the task.")